# MetaGO v2 — Research Notebook

Two-module decomposition:
- **Momentum module**: Lasso regression on EMA features + polynomial/interaction transforms → 1-candle H4 return prediction
- **Mean reversion module**: Ornstein-Uhlenbeck regression to weekly / monthly true opens → expected drift-to-mean

Gate: OU signal only fires when momentum module confirms direction.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import TimeSeriesSplit

from metalib.utils import load_hist_data
from metalib.indicators import (
    fit_ou_process_nb,
    calculate_half_life_nb,
    ols_tval_nb,
    build_lasso_cv,
    get_second_monday_open_ffill,
    get_last_monday_6pm_open_ffill,
)

pd.set_option('display.float_format', '{:.6f}'.format)
plt.rcParams['figure.figsize'] = (14, 5)

## 1. Data Loading & H4 Resampling

In [ ]:
SYMBOL   = 'eurusd'
YEARS    = [2019, 2020, 2021, 2022, 2023]
LOOKAHEAD = 1   # 1 H4 candle ahead

# ── Load & concatenate M1 data ────────────────────────────────────────────────
frames = [load_hist_data(SYMBOL, y) for y in YEARS]
m1 = pd.concat([f for f in frames if f is not None]).sort_index()
m1 = m1[~m1.index.duplicated()]
print(f"M1 rows: {len(m1):,}  |  {m1.index[0]} → {m1.index[-1]}")

# ── Resample to H4 ────────────────────────────────────────────────────────────
h4 = m1.resample('4H').agg({
    'open':   'first',
    'high':   'max',
    'low':    'min',
    'close':  'last',
    'volume': 'sum'
}).dropna()

print(f"H4 rows: {len(h4):,}")
h4.head(3)

## 2. Target: 1-Candle Ahead Log Return

In [ ]:
log_ret  = np.log(h4['close']).diff()
target   = log_ret.shift(-LOOKAHEAD).rename('fwd_ret')

# ── Annualised rolling volatility (used for vol-scaling) ─────────────────────
BARS_PER_YEAR = 252 * 6   # ~6 H4 bars per day
vol = log_ret.rolling(48).std() * np.sqrt(BARS_PER_YEAR)

print("Target stats:")
print(target.describe())

## 3. Feature Engineering

### 3a. Raw EMAs & Crossover Ratios

In [ ]:
EMA_SPANS = [5, 12, 24, 48, 96]

feats = pd.DataFrame(index=h4.index)
close = h4['close']

# ── Raw EMAs (price-normalised: ema/close - 1) ────────────────────────────────
emas = {}
for span in EMA_SPANS:
    e = close.ewm(span=span, adjust=False).mean()
    emas[span] = e
    feats[f'ema_{span}_norm'] = (e / close) - 1.0

# ── EMA crossover ratios (short/long - 1) ─────────────────────────────────────
cross_pairs = [(5, 12), (5, 24), (12, 24), (12, 48), (24, 96), (48, 96)]
for s, l in cross_pairs:
    feats[f'ema_cross_{s}_{l}'] = (emas[s] / emas[l]) - 1.0

print(f"EMA features: {feats.shape[1]}")
feats.head(3)

### 3b. T-stat of Vol-Scaled Returns Regressed on Time

Captures trend *strength* over rolling windows — a positive (negative) t-stat means
the return sequence has a statistically significant upward (downward) slope in the window.

In [ ]:
TSTAT_WINDOWS = [12, 24, 48, 96]   # H4 bars ≈ 2d, 4d, 8d, 16d

vol_scaled_ret = (log_ret / (vol / np.sqrt(BARS_PER_YEAR))).fillna(0.0)

for w in TSTAT_WINDOWS:
    tstat_col = np.full(len(h4), np.nan)
    vs_arr = vol_scaled_ret.values
    for i in range(w, len(vs_arr)):
        window_arr = vs_arr[i - w: i].astype(np.float64)
        if not np.any(np.isnan(window_arr)):
            tstat_col[i] = ols_tval_nb(window_arr)
    feats[f'tstat_vs_{w}'] = tstat_col

print(f"Total features after t-stat: {feats.shape[1]}")

### 3c. Polynomial & Interaction Transforms (degree 3)

Applied only to the base EMA features to keep dimensionality tractable before Lasso prunes them.

In [ ]:
POLY_DEGREE = 3

base_feat_names = [c for c in feats.columns if c.startswith('ema_')]
base_feats = feats[base_feat_names]

poly = PolynomialFeatures(degree=POLY_DEGREE, include_bias=False, interaction_only=False)
poly_arr = poly.fit_transform(base_feats.fillna(0.0))
poly_names = poly.get_feature_names_out(base_feat_names)

poly_df = pd.DataFrame(poly_arr, index=feats.index, columns=poly_names)
# Drop the degree-1 columns (already in feats) to avoid duplication
poly_df = poly_df.drop(columns=base_feat_names, errors='ignore')

feats = pd.concat([feats, poly_df], axis=1)
print(f"Total features after poly transforms: {feats.shape[1]}")

## 4. Momentum Module — Feature Predictiveness

### 4a. Information Coefficient (Spearman) per Feature

In [ ]:
# ── Align on clean rows ───────────────────────────────────────────────────────
combined = feats.join(target).dropna()
X_all = combined.drop(columns=['fwd_ret'])
y_all = combined['fwd_ret']

print(f"Clean rows: {len(combined):,}")

# ── IC = Spearman rank correlation ────────────────────────────────────────────
ics = {}
for col in X_all.columns:
    rho, pval = stats.spearmanr(X_all[col], y_all)
    ics[col] = {'IC': rho, 'p_value': pval}

ic_df = pd.DataFrame(ics).T.sort_values('IC', key=abs, ascending=False)

print("\nTop 20 features by |IC|:")
print(ic_df.head(20).to_string())

In [ ]:
# ── Restrict to statistically significant features (p < 0.05) ────────────────
sig_features = ic_df[ic_df['p_value'] < 0.05].index.tolist()
print(f"Significant features (p<0.05): {len(sig_features)} / {len(ic_df)}")

fig, ax = plt.subplots(figsize=(14, 4))
top = ic_df.head(40)
colors = ['steelblue' if v > 0 else 'tomato' for v in top['IC']]
ax.bar(range(len(top)), top['IC'], color=colors)
ax.set_xticks(range(len(top)))
ax.set_xticklabels(top.index, rotation=90, fontsize=7)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Top 40 Features by |IC| (Spearman vs fwd_ret)')
ax.set_ylabel('IC')
plt.tight_layout()
plt.show()

### 4b. Rolling IC — Stability Over Time

In [ ]:
ROLLING_IC_WINDOW = 252 * 2   # ~2 years of H4 bars  (252*6 bars/yr but daily grouping)

# Use only base EMA features + t-stat features (interpretable set)
monitor_feats = [c for c in feats.columns if 'tstat_vs' in c or c in base_feat_names]
monitor_feats = [f for f in monitor_feats if f in sig_features][:8]   # cap at 8

roll_ic = pd.DataFrame(index=combined.index)
for col in monitor_feats:
    roll_ic[col] = [
        stats.spearmanr(
            combined[col].iloc[max(0, i - ROLLING_IC_WINDOW): i],
            combined['fwd_ret'].iloc[max(0, i - ROLLING_IC_WINDOW): i]
        )[0] if i >= ROLLING_IC_WINDOW else np.nan
        for i in range(len(combined))
    ]

roll_ic.plot(figsize=(14, 5), title='Rolling IC (2-year window)', alpha=0.8)
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.ylabel('Spearman IC')
plt.tight_layout()
plt.show()

### 4c. Lasso CV Fit — Full Feature Set

In [ ]:
# ── Train on first 70%, evaluate on last 30% ─────────────────────────────────
split = int(len(combined) * 0.70)
X_train = X_all.iloc[:split].values
y_train = y_all.iloc[:split].values
X_test  = X_all.iloc[split:].values
y_test  = y_all.iloc[split:].values

print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

lasso_model, best_alpha = build_lasso_cv(
    X_train, y_train,
    normalize=True,
    fit_intercept=True,
    alphas=np.logspace(-8, 2, 200),
    cv=5
)
print(f"Best alpha: {best_alpha:.2e}")

In [ ]:
# ── Coefficients ──────────────────────────────────────────────────────────────
lasso_step = lasso_model.named_steps['lassocv']
coef_series = pd.Series(lasso_step.coef_, index=X_all.columns)
non_zero = coef_series[coef_series != 0].sort_values(key=abs, ascending=False)
print(f"\nNon-zero coefficients: {len(non_zero)} / {len(coef_series)}")
print(non_zero.head(20).to_string())

fig, ax = plt.subplots(figsize=(14, 4))
top_coef = non_zero.head(30)
colors = ['steelblue' if v > 0 else 'tomato' for v in top_coef]
ax.bar(range(len(top_coef)), top_coef.values, color=colors)
ax.set_xticks(range(len(top_coef)))
ax.set_xticklabels(top_coef.index, rotation=90, fontsize=7)
ax.set_title('Top 30 Lasso Coefficients (non-zero)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Out-of-sample IC ──────────────────────────────────────────────────────────
y_pred_test = lasso_model.predict(X_test)
oos_ic, oos_pval = stats.spearmanr(y_pred_test, y_test)
oos_r2 = 1 - np.sum((y_test - y_pred_test)**2) / np.sum((y_test - np.mean(y_test))**2)

print(f"OOS IC (Spearman): {oos_ic:.4f}  (p={oos_pval:.4f})")
print(f"OOS R²:            {oos_r2:.6f}")

### 4d. Walk-Forward Validation — Momentum Module

In [ ]:
WF_TRAIN_BARS = 252 * 6 * 2   # 2 years in-sample
WF_TEST_BARS  = 252 * 6 // 4  # 3 months out-of-sample per fold

X_np = X_all.values
y_np = y_all.values

wf_results = []
start = WF_TRAIN_BARS

while start + WF_TEST_BARS <= len(X_np):
    X_tr = X_np[start - WF_TRAIN_BARS: start]
    y_tr = y_np[start - WF_TRAIN_BARS: start]
    X_te = X_np[start: start + WF_TEST_BARS]
    y_te = y_np[start: start + WF_TEST_BARS]

    alphas = np.logspace(-8, 2, 100)
    pipe = make_pipeline(StandardScaler(), LassoCV(alphas=alphas, cv=3, max_iter=3000))
    pipe.fit(X_tr, y_tr)
    y_hat = pipe.predict(X_te)

    rho, pval = stats.spearmanr(y_hat, y_te)
    r2 = 1 - np.sum((y_te - y_hat)**2) / np.sum((y_te - np.mean(y_te))**2)
    wf_results.append({
        'fold_start': combined.index[start],
        'fold_end':   combined.index[min(start + WF_TEST_BARS - 1, len(combined)-1)],
        'IC':         rho,
        'p_value':    pval,
        'R2':         r2,
        'alpha':      pipe.named_steps['lassocv'].alpha_
    })
    start += WF_TEST_BARS

wf_df = pd.DataFrame(wf_results)
print(wf_df[['fold_start','fold_end','IC','p_value','R2','alpha']].to_string())
print(f"\nMean IC: {wf_df['IC'].mean():.4f}  |  % positive: {(wf_df['IC']>0).mean():.0%}")

## 5. Mean Reversion Module — Ornstein-Uhlenbeck

### 5a. Build OU Residuals: x_t = close_t − anchor_t

Discretised OU: `Δx_t = a + b·x_{t-1} + ε`  
→ θ = -b  (speed of reversion),  μ = -a/b  (long-run mean),  σ = std(ε)

In [ ]:
# ── True open anchors ─────────────────────────────────────────────────────────
true_open_monthly = get_second_monday_open_ffill(h4, h4.index)
true_open_weekly  = get_last_monday_6pm_open_ffill(h4, h4.index)

# Fill any leading NaN with first valid close
true_open_monthly = true_open_monthly.ffill().bfill()
true_open_weekly  = true_open_weekly.ffill().bfill()

# ── OU residuals ──────────────────────────────────────────────────────────────
x_monthly = (h4['close'] - true_open_monthly).rename('x_monthly')
x_weekly  = (h4['close'] - true_open_weekly ).rename('x_weekly')

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
x_monthly.plot(ax=axes[0], title='Monthly OU residual (close − 2nd-Monday-open)', color='steelblue')
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
x_weekly.plot(ax=axes[1], title='Weekly OU residual (close − last-Monday-6pm-open)', color='coral')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.show()

### 5b. Global OU Parameter Estimation

In [ ]:
def fit_ou_ols(x_series):
    """Fit discretised OU via OLS: Δx = a + b·x_{t-1} + ε"""
    x = x_series.dropna().values
    dx  = x[1:] - x[:-1]
    x_l = x[:-1]

    # OLS
    X_mat = np.column_stack([np.ones(len(x_l)), x_l])
    betas, _, _, _ = np.linalg.lstsq(X_mat, dx, rcond=None)
    a, b = betas
    resid = dx - (a + b * x_l)

    theta  = -b                               # speed of reversion (per bar)
    mu     = -a / b if b != 0 else 0.0       # long-run mean of residual
    sigma  = np.std(resid)                    # noise level
    half_life = np.log(2) / theta if theta > 0 else np.nan

    # t-stat on b (significance of mean reversion)
    se_b = sigma / (np.sqrt(np.sum((x_l - np.mean(x_l))**2)))
    tstat_b = b / se_b if se_b > 0 else np.nan

    return dict(theta=theta, mu=mu, sigma=sigma, half_life=half_life, tstat_b=tstat_b, a=a, b=b)


ou_monthly = fit_ou_ols(x_monthly)
ou_weekly  = fit_ou_ols(x_weekly)

print("OU Monthly:", {k: f"{v:.4f}" for k,v in ou_monthly.items()})
print("OU Weekly: ", {k: f"{v:.4f}" for k,v in ou_weekly.items()})

### 5c. OU Expected Return-to-Mean as a Predictor

`expected_drift_t = θ · (μ − x_t) · Δt`  (Δt = 1 bar here)

We test its IC against the 1-candle forward return.

In [ ]:
def ou_expected_drift(x_series, params):
    """θ·(μ − x_t), the OU drift signal at each bar."""
    return params['theta'] * (params['mu'] - x_series)


drift_monthly = ou_expected_drift(x_monthly, ou_monthly).rename('ou_drift_monthly')
drift_weekly  = ou_expected_drift(x_weekly,  ou_weekly ).rename('ou_drift_weekly')

# ── IC vs 1-candle forward return ────────────────────────────────────────────
ou_signals = pd.DataFrame({'ou_drift_monthly': drift_monthly, 'ou_drift_weekly': drift_weekly})
ou_combined = ou_signals.join(target).dropna()

for col in ['ou_drift_monthly', 'ou_drift_weekly']:
    rho, pval = stats.spearmanr(ou_combined[col], ou_combined['fwd_ret'])
    print(f"{col:25s}  IC={rho:+.4f}  p={pval:.4f}")

### 5d. Rolling OU Re-estimation — Parameter Stability

Each week (42 H4 bars) refit OU on trailing 6-month window (1092 bars).

In [ ]:
OU_FIT_WINDOW  = 6 * 21 * 6    # ~6 months of H4 bars
OU_REFIT_EVERY = 42            # refit every ~1 week

def rolling_ou_drift(x_series, fit_window=OU_FIT_WINDOW, refit_every=OU_REFIT_EVERY):
    """Re-estimate OU params on a rolling window and return drift series."""
    x_vals  = x_series.values
    n       = len(x_vals)
    drift   = np.full(n, np.nan)
    thetas, mus = [], []
    last_params = None

    for i in range(fit_window, n):
        if (i - fit_window) % refit_every == 0:
            window = x_vals[i - fit_window: i]
            if not np.any(np.isnan(window)):
                p = fit_ou_ols(pd.Series(window))
                if p['theta'] > 0:   # only use if mean-reverting
                    last_params = p
                    thetas.append(p['theta'])
                    mus.append(p['mu'])
        if last_params is not None:
            drift[i] = last_params['theta'] * (last_params['mu'] - x_vals[i])

    drift_series = pd.Series(drift, index=x_series.index, name=x_series.name + '_roll_drift')
    return drift_series, np.array(thetas), np.array(mus)


roll_drift_monthly, thetas_m, mus_m = rolling_ou_drift(x_monthly)
roll_drift_weekly,  thetas_w, mus_w = rolling_ou_drift(x_weekly)

fig, axes = plt.subplots(2, 2, figsize=(14, 7))

axes[0, 0].hist(thetas_m, bins=40, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Monthly θ (speed of reversion)')
axes[0, 1].hist(mus_m, bins=40, color='steelblue', edgecolor='white')
axes[0, 1].set_title('Monthly μ (long-run residual mean)')
axes[1, 0].hist(thetas_w, bins=40, color='coral', edgecolor='white')
axes[1, 0].set_title('Weekly θ')
axes[1, 1].hist(mus_w, bins=40, color='coral', edgecolor='white')
axes[1, 1].set_title('Weekly μ')
plt.suptitle('Rolling OU Parameter Distributions')
plt.tight_layout()
plt.show()

In [ ]:
# ── IC of rolling OU drift vs forward return ──────────────────────────────────
roll_ou_df = pd.DataFrame({
    'roll_drift_monthly': roll_drift_monthly,
    'roll_drift_weekly':  roll_drift_weekly
}).join(target).dropna()

for col in ['roll_drift_monthly', 'roll_drift_weekly']:
    rho, pval = stats.spearmanr(roll_ou_df[col], roll_ou_df['fwd_ret'])
    print(f"{col:30s}  IC={rho:+.4f}  p={pval:.4f}")

## 6. Combined Signal — OU Gated by Momentum

**Gate rule**: OU signal fires only when the momentum module's Lasso prediction  
has the *same sign* as the OU drift (i.e. both agree on direction).

We walk-forward both modules together and measure the gated signal quality.

In [ ]:
WF_TRAIN_BARS = 252 * 6 * 2
WF_TEST_BARS  = 252 * 6 // 4
OU_FIT_BARS   = OU_FIT_WINDOW

# ── Align all series on a common clean index ──────────────────────────────────
master = feats.join(target).join(roll_drift_monthly).join(roll_drift_weekly).dropna()
X_np   = master.drop(columns=['fwd_ret', 'x_monthly_roll_drift', 'x_weekly_roll_drift']).values
ou_m   = master['x_monthly_roll_drift'].values
ou_w   = master['x_weekly_roll_drift'].values
y_np   = master['fwd_ret'].values

comb_results = []
start = WF_TRAIN_BARS

while start + WF_TEST_BARS <= len(X_np):
    X_tr = X_np[start - WF_TRAIN_BARS: start]
    y_tr = y_np[start - WF_TRAIN_BARS: start]
    X_te = X_np[start: start + WF_TEST_BARS]
    y_te = y_np[start: start + WF_TEST_BARS]

    ou_m_te = ou_m[start: start + WF_TEST_BARS]
    ou_w_te = ou_w[start: start + WF_TEST_BARS]

    # ── Fit Lasso momentum ────────────────────────────────────────────────────
    pipe = make_pipeline(StandardScaler(), LassoCV(alphas=np.logspace(-8, 2, 100), cv=3, max_iter=3000))
    pipe.fit(X_tr, y_tr)
    mom_pred = pipe.predict(X_te)   # momentum signal

    # ── OU signal: use monthly unless weekly is stronger ─────────────────────
    ou_pred = np.where(np.abs(ou_w_te) > np.abs(ou_m_te), ou_w_te, ou_m_te)

    # ── Gate: combined only when both agree on direction ─────────────────────
    gate = np.sign(mom_pred) == np.sign(ou_pred)
    gated_pred = np.where(gate, ou_pred, 0.0)

    n_active = gate.sum()
    if n_active > 10:
        rho_mom,  _     = stats.spearmanr(mom_pred,   y_te)
        rho_ou,   _     = stats.spearmanr(ou_pred,    y_te)
        rho_gate, pgate = stats.spearmanr(gated_pred[gate], y_te[gate])
    else:
        rho_mom = rho_ou = rho_gate = pgate = np.nan

    comb_results.append({
        'fold_start':    master.index[start],
        'IC_momentum':   rho_mom,
        'IC_ou':         rho_ou,
        'IC_gated':      rho_gate,
        'p_gated':       pgate,
        'active_pct':    gate.mean()
    })
    start += WF_TEST_BARS

comb_df = pd.DataFrame(comb_results).set_index('fold_start')
print(comb_df.to_string())
print(f"\nMean gated IC: {comb_df['IC_gated'].mean():.4f}")
print(f"Mean active  : {comb_df['active_pct'].mean():.1%}")

In [ ]:
# ── Visual comparison ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(comb_df))
w = 0.25

ax.bar([i - w for i in x], comb_df['IC_momentum'], width=w, label='Momentum', color='steelblue', alpha=0.8)
ax.bar([i      for i in x], comb_df['IC_ou'],       width=w, label='OU',       color='coral',    alpha=0.8)
ax.bar([i + w for i in x], comb_df['IC_gated'],     width=w, label='Gated',    color='seagreen',  alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels([str(d.date()) for d in comb_df.index], rotation=45)
ax.set_title('Walk-Forward IC: Momentum vs OU vs Gated Combined')
ax.set_ylabel('Spearman IC')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Summary & Research Conclusions

Key findings to carry into the v2 implementation:

In [ ]:
print("=" * 60)
print("MOMENTUM MODULE")
print(f"  Significant features (p<0.05) : {len(sig_features):3d}")
print(f"  Non-zero Lasso coefs           : {(lasso_step.coef_ != 0).sum():3d}")
print(f"  Walk-forward mean IC           : {wf_df['IC'].mean():+.4f}")
print(f"  WF folds with IC > 0           : {(wf_df['IC'] > 0).sum()} / {len(wf_df)}")

print()
print("MEAN REVERSION MODULE (global OU)")
print(f"  Monthly θ (speed)  : {ou_monthly['theta']:.4f}")
print(f"  Monthly half-life  : {ou_monthly['half_life']:.1f} bars")
print(f"  Monthly t-stat(b)  : {ou_monthly['tstat_b']:.2f}")
print(f"  Weekly  θ (speed)  : {ou_weekly['theta']:.4f}")
print(f"  Weekly  half-life  : {ou_weekly['half_life']:.1f} bars")
print(f"  Weekly  t-stat(b)  : {ou_weekly['tstat_b']:.2f}")

print()
print("COMBINED GATED SIGNAL")
print(f"  Walk-forward mean IC    : {comb_df['IC_gated'].mean():+.4f}")
print(f"  WF folds with IC > 0    : {(comb_df['IC_gated'] > 0).sum()} / {len(comb_df)}")
print(f"  Mean active fraction    : {comb_df['active_pct'].mean():.1%}")
print("=" * 60)